# Module 2 -- Step 1: LLM Gateway Architecture Overview

This step provides a conceptual overview of the **LLM Gateway** architecture.
You will learn why enterprises need a centralized gateway for LLM access,
what components make up the solution, and how it integrates with the rest of
the Agentic AI platform.

**No infrastructure is deployed in this step.** Deployment happens in Step 2.

## Why Enterprises Need an LLM Gateway

As teams adopt large language models, several operational challenges emerge quickly:

| Challenge | Description |
|---|---|
| **Credential sprawl** | Every team creates its own IAM roles, API keys, and Bedrock access patterns. Secrets leak into notebooks, repos, and CI pipelines. |
| **Cost governance** | Without centralized metering, monthly Bedrock bills become unpredictable. There is no per-team or per-project budget enforcement. |
| **Multi-model routing** | Teams hard-code model IDs. Switching from Claude to Llama or upgrading model versions requires code changes across dozens of services. |
| **Observability gaps** | Token usage, latency percentiles, and error rates are scattered across individual service logs with no unified view. |
| **Security and compliance** | There is no single control plane to enforce guardrails, content filtering, or audit logging for all LLM traffic. |

An **LLM Gateway** solves these problems by providing a single, governed entry point
for all LLM interactions across the enterprise. Teams authenticate with virtual keys,
the gateway routes requests to the correct model provider, and every call is metered,
logged, and budget-checked automatically.

## Architecture Components

The LLM Gateway is deployed entirely within your AWS account using the following components:

### Networking

- **VPC** -- Dedicated VPC with public and private subnets across two Availability Zones.
- **Application Load Balancer (ALB)** -- Internet-facing ALB that terminates HTTPS and forwards traffic to the ECS service.
- **API Gateway** -- Optional HTTP API in front of the ALB for additional request throttling and API key management.

### Compute

- **ECS Fargate** -- Runs the LiteLLM proxy as a serverless container. No EC2 instances to manage.
- **PostgreSQL (Fargate sidecar)** -- Lightweight PostgreSQL container for LiteLLM's internal state: virtual keys, spend tracking, and team budgets.
- **EFS (Elastic File System)** -- Provides persistent storage for the PostgreSQL data directory so state survives task restarts.

### Security

- **IAM Roles** -- Task execution role (ECR pull, CloudWatch Logs) and task role (Bedrock InvokeModel, Secrets Manager read).
- **Secrets Manager** -- Stores the LiteLLM admin key and PostgreSQL credentials. Never hard-coded.
- **Security Groups** -- ALB accepts inbound HTTPS; ECS tasks only accept traffic from the ALB security group.

### Observability

- **CloudWatch Logs** -- All LiteLLM proxy logs stream to a dedicated log group.
- **CloudWatch Metrics** -- ECS service metrics (CPU, memory) plus custom LiteLLM metrics for token throughput and latency.

### Data Flow

```
Client / Agent
      |
      v
  API Gateway (optional)
      |
      v
  ALB (HTTPS)
      |
      v
  ECS Fargate Task
  +---------------------------+
  | LiteLLM Proxy (port 4000) |----> Amazon Bedrock (Claude, Llama, Titan, ...)
  | PostgreSQL  (port 5432)   |
  +---------------------------+
      |
      v
  EFS (persistent DB storage)
```

## LiteLLM: OpenAI-Compatible API for 70+ Bedrock Models

[LiteLLM](https://github.com/BerriAI/litellm) is an open-source proxy that exposes an
**OpenAI-compatible API** (`/chat/completions`, `/embeddings`, `/images/generations`)
while routing requests to 100+ LLM providers behind the scenes.

For this workshop, LiteLLM is configured to route to **Amazon Bedrock** models using
the `bedrock/` prefix. Examples:

| LiteLLM model string | Bedrock model |
|---|---|
| `bedrock/anthropic.claude-sonnet-4-20250514` | Claude Sonnet 4 |
| `bedrock/anthropic.claude-haiku-4-20250514` | Claude Haiku 4 |
| `bedrock/meta.llama3-1-70b-instruct-v1:0` | Llama 3.1 70B |
| `bedrock/amazon.titan-text-premier-v1:0` | Titan Text Premier |
| `bedrock/amazon.titan-embed-text-v2:0` | Titan Embeddings v2 |

Key benefits of this approach:

- **Single API contract** -- Every client sends OpenAI-format requests regardless of the backend model.
- **Model fallbacks** -- Configure primary and fallback models. If Claude is throttled, fall back to Llama automatically.
- **Load balancing** -- Distribute requests across multiple model deployments.
- **Spend tracking** -- Every request is logged with token counts, cost, and the virtual key that made it.

The cell below queries your account to see which Bedrock foundation models are currently available.

In [ ]:
import boto3

AWS_REGION = boto3.session.Session().region_name or "us-west-2"
bedrock = boto3.client("bedrock", region_name=AWS_REGION)
models = bedrock.list_foundation_models()["modelSummaries"]

print(f"Region: {AWS_REGION}")
print(f"Total Bedrock models available: {len(models)}\n")
for m in sorted(models, key=lambda x: x["modelId"])[:15]:
    print(f"  {m['modelId']:55s}  {m['providerName']}")
print(f"\n  ... and {len(models)-15} more")

## Virtual Keys and Team-Based Budgets

LiteLLM's **virtual key** system is the foundation of the gateway's multi-tenant cost governance.
Instead of distributing raw AWS credentials, the gateway administrator issues virtual API keys
that behave like OpenAI API keys but are fully managed by the proxy.

### How It Works

1. **Teams** -- Create logical teams (e.g., `fraud-detection`, `customer-support`, `data-science`).
   Each team has a monthly budget ceiling.
2. **Virtual keys** -- Generate one or more keys per team. Each key inherits the team's budget
   and can optionally have its own per-key spend limit.
3. **Model access control** -- Restrict which models a team or key can use. For example,
   the `data-science` team may access Claude and Llama, while `customer-support` is limited to
   Haiku for cost reasons.
4. **Spend tracking** -- Every request records the token count and estimated cost. When a team
   hits its budget, subsequent requests are rejected with a clear error.

### Hands-on in Step 3

You will create teams and virtual keys hands-on in **Step 3: Virtual Keys and Teams**, using the same `/team/new` and `/key/generate` admin endpoints.


## Integration with Strands Agents

The Agentic AI platform uses the [Strands Agents SDK](https://github.com/strands-agents/sdk-python)
to build autonomous agents. Strands natively supports a **LiteLLMModel** provider, which means
agents can route all their LLM calls through the gateway with minimal configuration.

### How Agents Connect

```python
from strands import Agent
from strands.models.litellm import LiteLLMModel

# Point the agent at the LLM Gateway.
# Always use HTTPS endpoints (the workshop gateway URL is the
# HTTPS API Gateway endpoint from the stack outputs).
model = LiteLLMModel(
    client_args={
        "api_key": "sk-...",  # Virtual key issued by the gateway
        "api_base": "https://<GATEWAY_URL>",
        "use_litellm_proxy": True,
    },
    model_id="claude-sonnet",  # model alias registered on the gateway
)

agent = Agent(model=model)
response = agent("Summarize the latest fraud alerts.")
```

### Benefits for Agent Workloads

- **Automatic retries and fallbacks** -- If the primary model is throttled, the gateway
  falls back to a secondary model without the agent needing any retry logic.
- **Unified spend tracking** -- Every agent call is attributed to a team and key,
  giving operators full visibility into per-agent costs.
- **Model upgrades without code changes** -- When a new model version is released,
  update the gateway routing config. All agents pick up the change immediately.
- **Guardrails enforcement** -- Content filtering and compliance rules are applied at
  the gateway layer, ensuring all agents respect enterprise policies.

---

## Next Step

Now that you understand the architecture, proceed to
**Step 2: Deploy the LLM Gateway** where you will provision the full stack
using CloudFormation and verify the gateway is operational.

Open the next notebook: `step-2-deploy.ipynb`